# Day 3 — Pydantic v2: Validate at the Boundary

---

A backend's #1 source of bugs is **bad input**. The rule:

> *Validate at the boundary. Trust the inside.*

**Pydantic** is the validation layer FastAPI is built on. Declare what data should look like as a class — Pydantic parses, coerces, and rejects everything else with a clear error.


In [ ]:
!pip install pydantic fastapi uvicorn


## Your First `BaseModel`

Subclass `BaseModel` and declare fields with type hints. Pydantic does the rest.


In [1]:
from pydantic import BaseModel, ValidationError

class User(BaseModel):
    name: str
    age: int

# Valid input — coerces "30" → 30 automatically
u = User(name="Alice", age="30")
print(u)
print(type(u.age).__name__)

# Invalid input → ValidationError
try:
    User(name="Bob", age="thirty")
except ValidationError as e:
    print("\nGOT ERROR:\n", e)


name='Alice' age=30
int

GOT ERROR:
 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing


## Field Types

Pydantic understands all the usual Python types:


In [ ]:
age="uday"
type(age).__name__

class Profile(BaseModel):
    name: str
    age: int
    height: float
    is_active: bool
    tags: list[int]
    scores: dict[str, int]

p = Profile(
    name="Alice",
    age=30,
    height=1.75,
    is_active=True,
    tags=[1, 2, 3],
    scores={"math": 90, "physics": 85},
)
print(p)


name='Alice' age=30 height=1.75 is_active=True tags=[1, 2, 3] scores={'math': 90, 'physics': 85}


## `Field()` — Constraints & Metadata

Use `Field(...)` to add validation rules, a default, and a description.
The `...` (Ellipsis) literally means "required".


In [11]:
from pydantic import Field

class Item(BaseModel):
    name: str = Field(..., min_length=3, max_length=50, description="Item name",pattern=r"^[a-zA-Z0-9_]+$")
    price: float = Field(..., gt=0, description="Must be > 0")
    quantity: int = Field(default=0, ge=0, description="Must be >= 0")

print(Item(name="Pen", price=2.5, quantity=10))             # quantity defaults to 0

try:
    Item(name="Pe", price=-1, quantity=-3)
except ValidationError as e:
    print("\nGOT ERROR:")
    for err in e.errors():
        print(" ", err["loc"], "->", err["msg"])


name='Pen' price=2.5 quantity=10

GOT ERROR:
  ('name',) -> String should have at least 3 characters
  ('price',) -> Input should be greater than 0
  ('quantity',) -> Input should be greater than or equal to 0


## Optional Fields

Python's `X | None = None` is the modern way to say "optional".


In [7]:
class Customer(BaseModel):
    name: str
    email: str | None = None      # optional
    age: int | None = None

print(Customer(name="Alice"))
print(Customer(name="Bob", email="bob@x.com"))


name='Alice' email=None age=None
name='Bob' email='bob@x.com' age=None


## Field Constraints — Reference Table

| Constraint | Meaning | Works on |
|------------|---------|----------|
| `min_length` / `max_length` | string/list length | str, list, bytes |
| `gt` / `ge` | greater than / ≥ | numeric |
| `lt` / `le` | less than / ≤ | numeric |
| `pattern` | regex match | str |
| `default` | default value | any |
| `description` | shows up in `/docs` | any |


## Nested Models

Models can contain other models — Pydantic validates the whole tree.


In [6]:
class Category(BaseModel):
    name: str = Field(..., min_length=2, max_length=30)

class Product(BaseModel):
    name: str = Field(..., min_length=3)
    price: float = Field(..., gt=0)
    category: Category

p = Product(name="Pen", price=2.5, category={"name": "Stationery"})
print(p)
print("Category name:", p.category.name)


name='Pen' price=2.5 category=Category(name='Stationery')
Category name: Stationery


## Pydantic + FastAPI

When a FastAPI endpoint takes a model as a parameter, FastAPI:

1. Reads the request body as JSON
2. Validates it against the model
3. Either calls your function with a parsed object **or** returns `422` with the errors

No manual parsing. No defensive `if`s. The function body only runs on **valid** input.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()

@app.post("/items")
def create_item(item: Item):
    return {"received": item.model_dump()}
    


client = TestClient(app)

# Valid request → 200
print(client.post("/items", json={"name": "Pen", "price": 2.5}).json())

# Invalid request → 422 with structured errors
bad = client.post("/items", json={"name": "X", "price": -1})
print("\nStatus:", bad.status_code)
print("Errors:", bad.json())


{'received': {'name': 'Pen', 'price': 2.5, 'quantity': 0}}

Status: 422
Errors: {'detail': [{'type': 'string_too_short', 'loc': ['body', 'name'], 'msg': 'String should have at least 3 characters', 'input': 'X', 'ctx': {'min_length': 3}}, {'type': 'greater_than', 'loc': ['body', 'price'], 'msg': 'Input should be greater than 0', 'input': -1, 'ctx': {'gt': 0.0}}]}


## `response_model=` — Validate Output Too

Declaring a `response_model` makes FastAPI:

- **Strip extra fields** that aren't in the model (great for hiding internal state)
- Validate the response shape (catches bugs in *your* code)
- Use the model in `/docs` for the response schema


In [10]:
class PublicItem(BaseModel):
    name: str
    price: float

@app.post("/safe-items", response_model=PublicItem)
def create_safe_item(item: Item):
    # We return *extra* fields (quantity, an internal id) — FastAPI strips them
    return {"name": item.name, "price": item.price,
            "quantity": item.quantity, "_internal_id": "secret"}

print(client.post("/safe-items", json={"name": "Pen", "price": 2.5, "quantity": 5}).json())


{'name': 'Pen', 'price': 2.5}


## `.model_dump()` — Model → dict

In **Pydantic v2** the method is `.model_dump()`.
(In v1 it was `.dict()`. You'll still see that in old tutorials — don't use it in new code.)


In [15]:
item = Item(name="Pen", price=2.5, quantity=10)
print(item.model_dump())            # dict
print(item.model_dump_json())       # JSON string


{'name': 'Pen', 'price': 2.5, 'quantity': 10}
{"name":"Pen","price":2.5,"quantity":10}


## A Peek at `@field_validator`

When `Field(...)` constraints aren't enough, write a custom validator. We'll go deep on this in the assignments — here's a one-line preview:

```python
from pydantic import field_validator

class Account(BaseModel):
    password: str

    @field_validator("password")
    @classmethod
    def must_have_digit(cls, v: str) -> str:
        if not any(c.isdigit() for c in v):
            raise ValueError("password must contain at least one digit")
        return v
```


## Recap

- `BaseModel` + type hints = automatic parsing + validation.
- `Field(...)` adds constraints: `min_length`, `max_length`, `gt`, `ge`, `lt`, `le`, `pattern`.
- Optional fields: `X | None = None`.
- Models nest — validation goes all the way down.
- A FastAPI endpoint typed with a model gives you **free** 422 on bad JSON.
- `response_model=` strips extras and validates output.
- `.model_dump()` in v2 (not `.dict()`).
- Custom rules → `@field_validator` (coming up in the assignments).
